In [38]:
import importlib
import sys
sys.path.append("../utils")
from specaugment_wrapper import MaybeSpecAugment

import images_dataset2
from images_dataset2 import ImagesDataset
importlib.reload(images_dataset2)
from images_dataset2 import ImagesDataset

import audio
importlib.reload(audio)
    
images = ImagesDataset(
    audio_dir="curated",
    test_audio_dir="test",
    labels_csv="train_curated.csv",
)

#images.split_train_val()
#images.split_audios_train(5) # to train_curated_split
#images.augment_audios_train()

In [39]:
import comet_ml
from comet_ml import Experiment

train_dataset, val_dataset, mlb = images.get_train_and_val_dataset(use_augmented_audios=True)

train_curated_augmented_spectrograms
Found 7952 files.
Identified 80 unique labels for 7952 files.
Found 994 files.
Identified 80 unique labels for 994 files.


In [42]:
api_key = "mgGsU36fCBuXeX37BIlnf04Yn"
experiment = Experiment(
    api_key,
    workspace='lab2',
    project_name='baseline_hmade',
    auto_histogram_weight_logging=True,
    auto_histogram_gradient_logging=True,
    auto_histogram_activation_logging=True,
)
experiment.add_tag('hmade')
experiment.set_name("Augmented no regula")

COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: sklearn, tensorflow, keras.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/lab2/baseline-hmade/e5c8554d4dee4600b8453409bbdc9e96



In [40]:
def ConvBlock(filters, downsample=False):
    convolution_1 = tf.keras.Sequential([
            tf.keras.layers.Conv2D(filters, (3, 3), padding='same', use_bias=False),
            tf.keras.layers.MaxPooling2D(),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.ReLU(),
    ])
    convolution_2 = tf.keras.Sequential([
            tf.keras.layers.Conv2D(filters, (2, 2), padding='same', use_bias=False),
            tf.keras.layers.MaxPooling2D(),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.ReLU()])
    convolution_full = tf.keras.Sequential([
                convolution_1,
                convolution_2,])
    return convolution_full

def create_model(num_classes=80,optimizer='Nadam',learning_rate=1e-2,image_size=(224, 224)):
    spec_augment = MaybeSpecAugment(
                freq_mask_param=5,
                time_mask_param=10,
                n_freq_mask=5,
                n_time_mask=3,
                mask_value=-100,
            )
    inputs = tf.keras.Input(shape=image_size + (3,))
    preproc = tf.keras.Sequential([
            #spec_augment,
            tf.keras.layers.Rescaling(scale=1./127.5, offset=-1),
            ConvBlock(filters=64),
            ConvBlock(filters=128),
            tf.keras.layers.Dropout(0),
            tf.keras.layers.GlobalAveragePooling2D(),
    ])
    
    predict = tf.keras.layers.Dense(num_classes, activation="sigmoid")
    
    x = preproc(inputs)
    outputs = predict(x)
    optimizer = tf.keras.optimizers.Nadam(learning_rate=learning_rate)
    model = tf.keras.models.Model(inputs, outputs)
    model.compile(loss="binary_crossentropy", optimizer=optimizer, metrics=[LwLrap(num_classes)])
    return model

In [43]:
import tensorflow as tf
import keras_tuner as kt
from lwlrap import calculate_per_class_lwlrap


import lwlrap
from lwlrap import LwLrap

importlib.reload(lwlrap)

params={'batch_size':16,
        'epochs':15,
        'epochs_fine_tune':12,
        'learning_rate':1e-2,
        'optimizer':"Nadam",
}

early = tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)
checkpoint = tf.keras.callbacks.ModelCheckpoint("baseline/best_hmade.keras", monitor='val_lwlrap', verbose=1, save_best_only=True, mode='max')

#model = create_model()
#with experiment.train():
#    history = model.fit(train_dataset, epochs=params["epochs"], validation_data=val_dataset, callbacks=[early,checkpoint])

with experiment.test():
    loss, lwlrap = model.evaluate(val_dataset)
    
    y_preds = model.predict(val_dataset)
    y_truth = images.val_labels
    per_class_lwlrap, weights_per_class = calculate_per_class_lwlrap(y_truth, y_preds)
    
    metrics = {
        'loss':loss,
        'lwlrap':lwlrap,
        'per_class_lwlrap':per_class_lwlrap
    }
    experiment.log_metrics(metrics)

experiment.log_parameters(params)
experiment.log_curve("lwlrap train", range(len(history.history["lwlrap"])), history.history["lwlrap"])
experiment.log_curve("lwlrap val", range(len(history.history["val_lwlrap"])), history.history["val_lwlrap"])
experiment.log_curve("loss train", range(len(history.history["loss"])), history.history["loss"])
experiment.log_curve("loss val", range(len(history.history["val_loss"])), history.history["val_loss"])

experiment.end()

32/32 ━━━━━━━━━━━━━━━━━━━━ 11s 326ms/step - loss: 0.0471 - lwlrap: 0.5879
32/32 ━━━━━━━━━━━━━━━━━━━━ 11s 332ms/step


COMET WARNING: Cannot safely convert array([0.05562152, 0.04307302, 0.05591224, 0.20438937, 0.07211072,
       0.09232837, 0.03016535, 0.13982845, 0.04180234, 0.0237602 ,
       0.05079827, 0.02764325, 0.04905656, 0.16726264, 0.11101364,
       0.100552  , 0.08895232, 0.04447201, 0.04308521, 0.05761114,
       0.04027884, 0.08570046, 0.01747753, 0.19029463, 0.08061891,
       0.05000763, 0.05957037, 0.0361944 , 0.09222451, 0.11164746,
       0.09219771, 0.0339832 , 0.05281357, 0.10178735, 0.06662337,
       0.03914287, 0.03590878, 0.01324155, 0.08814261, 0.03303043,
       0.02176631, 0.02495188, 0.03524928, 0.02388891, 0.02681557,
       0.05898806, 0.12510277, 0.11045043, 0.05790534, 0.06035889,
       0.04188819, 0.14580795, 0.07651427, 0.06675561, 0.11684159,
       0.10500534, 0.08771611, 0.02972111, 0.04040247, 0.0217083 ,
       0.08711158, 0.06694314, 0.04395654, 0.04663953, 0.07653251,
       0.11459353, 0.13773927, 0.05693171, 0.1024274 , 0.02324746,
       0.02257947, 0.0645